## Feature Engineering

Essa é a etapa criativa. Os campos que existem transformam em representações que o modelo consegue usar de forma eficiente.

In [1]:
# libs
import pandas
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix

movies_clean = pandas.read_csv("../data/processed/movies_clean.csv")
movies = pandas.read_csv("../data/raw/ml-small/movies.csv")
ratings = pandas.read_csv("../data/raw/ml-small/ratings.csv")
tags = pandas.read_csv("../data/raw/ml-small/tags.csv")
links = pandas.read_csv("../data/raw/ml-small/links.csv")

- **Gêneros → multi-hot**

Cada gênero vira uma coluna binária. Um filme de Crime e Thriller fica com 1 nessas duas colunas e 0 em todas as outras.
Isso permite calcular similaridade entre filmes por gênero de forma direta.

In [2]:
# Processar gêneros (multi-hot)
movies["genres_list"] = movies["genres"].str.split("|")
mlb_genres = MultiLabelBinarizer()
genre_features = mlb_genres.fit_transform(movies["genres_list"])

genre_sparse = csr_matrix(genre_features.astype(float))

# print(movies["genres_list"])

- **Ano de lançamento → feature numérica**

O ano extraído do título vira um número normalizado. Usuários tendem a ter preferências por épocas.

In [3]:
# Normaliza para o intervalo [0, 1]
scaler = MinMaxScaler()
year_feature = scaler.fit_transform(movies_clean[["year"]])

year_sparse = csr_matrix(year_feature.astype(float))

- **Tags → texto agregado por filme**

As tags vêm por usuário e por filme. `userId=18, movieId=296, tag="nonlinear timeline"`.
Você agrega todas as tags de um filme em um único texto por filme. 
Esse texto é extremamente valioso porque é o vocabulário que os próprios usuários usam para descrever o filme — muito mais rico que a sinopse oficial.

Cuidado na normalização: "Tarantino", "tarantino" e "TARANTINO".

In [4]:
# Agregar tags por filme como texto
tags_agrupadas = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.str.lower()))
    .reset_index()
    .rename(columns={"tag": "tags_texto"})
)
movies = movies.merge(tags_agrupadas, on="movieId", how="left")
movies["tags_texto"] = movies["tags_texto"].fillna("")

# Construir corpus de texto (sinopse + tags + título)
# Se tiver TMDB: overview + keywords + tags
movies["corpus"] = (
    movies["title"] + " " +
    movies["tags_texto"]
    # + tmdb_df["overview"] + " " + tmdb_df["keywords_texto"]
)

# TF-IDF no corpus
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2)  # captura bigramas como "time travel"
)
text_features = tfidf.fit_transform(movies["corpus"])

- **Timestamp dos ratings → peso temporal**

Ratings mais recentes do usuário revelam o gosto atual, não o histórico distante. 
Um usuário que avaliou 200 filmes há 10 anos e 5 filmes na semana passada — os 5 recentes dizem mais sobre o que ele quer agora. 
Podemos aplicar um fator de decaimento temporal que aumenta o peso dos ratings mais novos.

In [ ]:
# a definir...

In [ ]:
# Combinar features com pesos
features_finais = hstack([
    text_features * 0.5,  # texto: sinopse + tags
    genre_sparse  * 0.5,  # gêneros
    year_sparse   * 0.2,  # ano
])